In [ ]:
ROWS = 5_000_000

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "1000")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sales = (spark.range(ROWS)
         .withColumn("customer_id", F.when(F.rand(42) < 0.7, F.lit(1))
                                     .otherwise((F.rand(7) * 100000).cast("int")))
         .withColumn("product_id", (F.rand(3) * 500).cast("int"))
         .withColumn("amount", F.round(F.rand(11) * 1000, 2))
         .withColumn("ts", F.timestamp_seconds(F.lit(1767225600) + (F.col("id") % 31536000))))
sales.repartition(400).write.mode("overwrite").format("delta").saveAsTable("perf_demo_sales")

products = (spark.range(500).withColumnRenamed("id", "product_id")
            .crossJoin(spark.range(3).withColumnRenamed("id", "version"))
            .withColumn("category", F.concat(F.lit("cat_"), (F.col("product_id") % 20).cast("string")))
            .withColumn("list_price", F.round(F.rand(5) * 100, 2)))
products.write.mode("overwrite").format("delta").saveAsTable("perf_demo_products")

customers = (spark.range(100001).withColumnRenamed("id", "customer_id")
             .withColumn("segment", F.when(F.col("customer_id") % 3 == 0, "retail")
                                     .when(F.col("customer_id") % 3 == 1, "smb").otherwise("enterprise")))
customers.write.mode("overwrite").format("delta").saveAsTable("perf_demo_customers")

In [ ]:
s = spark.read.table("perf_demo_sales")
p = spark.read.table("perf_demo_products")
c = spark.read.table("perf_demo_customers")

enriched = (s.join(p, "product_id")
             .join(c, "customer_id")
             .dropDuplicates(["id"]))

summary = (enriched.groupBy("customer_id", "segment", "category")
           .agg(F.sum("amount").alias("total"),
                F.countDistinct("product_id").alias("products"))
           .orderBy(F.desc("total")))
summary.write.mode("overwrite").format("delta").saveAsTable("perf_demo_customer_summary")

In [ ]:
w = Window.partitionBy("customer_id").orderBy("ts")

ranked = (s.join(p, "product_id")
           .join(c, "customer_id")
           .dropDuplicates(["id"])
           .withColumn("running_total", F.sum("amount").over(w))
           .withColumn("rn", F.row_number().over(w)))

ranked.filter(F.col("rn") <= 5).write.mode("overwrite").format("delta").saveAsTable("perf_demo_first_purchases")